# UNI-24 · BUSI: данные и EDA в Google Colab

[Открыть в Google Colab](https://colab.research.google.com/github/carbon77/breast-cancer-classification/blob/codex/uni-24-data-pipeline/notebooks/01_busi_data_eda.ipynb)

Самодостаточный ноутбук: **весь код находится здесь**. Не требуются pip install, requirements.txt, клонирование репозитория или дополнительные .py-файлы. Используются библиотеки стандартного runtime Colab; версии фактического окружения сохраняются в артефактах.

Источник: [Breast Ultrasound Images Dataset](https://www.kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset). Классы: **0 normal, 1 benign, 2 malignant**. Ожидаемые 133/437/210 — ориентир; фактические числа берутся из аудита. Маски исключены из входа классификатора.

Этот notebook реализует E0. Обучение baseline CNN, transfer CNN и ViT и получение метрик выполняются позже. Здесь нет результатов запуска; проверку настоящего BUSI выполняет автор задачи.

**Порядок:** открыть notebook → проверить настройки → Run all → разрешить подключение Google Drive → просмотреть отчёты и графики. По умолчанию split и отчёты сохраняются в Drive и используются повторно между сессиями.

## 1. Соответствие UNI-23 и его подтикетам

| Тикет | Зафиксированное решение | Что делает этот notebook |
| --- | --- | --- |
| [UNI-31](https://linear.app/zakat-88/issue/UNI-31) | BUSI, 3 класса, обучение отдельно от FastAPI/Streamlit, воспроизводимый preprocessing | Выполняет подготовку данных в Colab; экспортирует preprocessing для будущего serving; маски не использует |
| [UNI-32](https://linear.app/zakat-88/issue/UNI-32) | macro-F1, per-class metrics, фиксированные пороги, test только для итоговой оценки | Не вычисляет model metrics и не меняет пороги; сохраняет ограничения и контракт будущих экспериментов |
| [UNI-33](https://linear.app/zakat-88/issue/UNI-33) | E0 → baseline CNN → transfer CNN → ViT; один split 70/15/15, seed 42, pHash≤4; train-only аугментации | Реализует E0 и общий input pipeline; split остаётся тем же для seed обучения 42/43/44 |

UNI-37/38/39 отмечены Duplicate; источники требований — UNI-31/32/33. Изменение формата реализации на один .ipynb не меняет критерии качества и план моделей.

Для pretrained-моделей параметры нормализации должны совпадать с конкретными весами. Код здесь предоставляет ImageNet preset; архитектуры и веса будут выбраны в задачах обучения.

In [ ]:
# Импорты из стандартного окружения Colab; пакеты не устанавливаются.
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import json
import shutil
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

# Все функции подготовки определены в следующих ячейках этого notebook.


### Функции аудита, группировки и input pipeline

Эти ячейки только определяют функции. Все определения находятся внутри notebook; внешние модули не импортируются.

In [ ]:
"""Dataset audit, duplicate-safe splits and shared PyTorch input pipeline.

Paths in manifests are relative to DATA_ROOT, never machine-specific paths.
No download or preparation happens on import.
"""

import hashlib
import importlib.metadata
import json
import platform
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from scipy.fft import dctn
from sklearn.model_selection import GroupShuffleSplit

CLASSES = ("normal", "benign", "malignant")
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASSES)}
SEED = 42
IMAGE_SIZE = 224
PIPELINE_VERSION = "uni24-colab-v2"
DATASET_SLUG = "aryashah2k/breast-ultrasound-images-dataset"
MASK_PATTERN = re.compile(r"_mask.*$", re.IGNORECASE)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def environment():
    packages = ("numpy", "pandas", "Pillow", "scipy", "scikit-learn", "matplotlib", "torch", "torchvision")
    return {"python": platform.python_version(), "packages": {
        name: importlib.metadata.version(name) for name in packages
    }}


def find_data_root(path):
    """Accept either Dataset_BUSI_with_GT or its Kaggle wrapper directory."""
    path = Path(path).resolve()
    candidates = [path] + sorted(p for p in path.rglob("Dataset_BUSI_with_GT") if p.is_dir())
    matches = [p for p in candidates if all((p / c).is_dir() for c in CLASSES)]
    if len(matches) != 1:
        raise ValueError(f"Expected one BUSI root with {CLASSES}, found {matches}")
    return matches[0]


def phash(image):
    """64-bit pHash: grayscale 32x32 -> 2D DCT -> 8x8 median threshold.

    Matches the conventional ImageHash pHash construction (including DC).
    """
    pixels = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.LANCZOS), dtype=float)
    low = dctn(pixels, type=2)[:8, :8]
    bits = low > np.median(low)
    return f"{int(''.join('1' if b else '0' for b in bits.ravel()), 2):016x}"


def scan_dataset(root):
    """Inventory ALL files, masks included; quarantine invalid original images."""
    root = find_data_root(root)
    rows = []
    for path in sorted(p for p in root.rglob("*") if p.is_file()):
        rel = path.relative_to(root)
        label = rel.parts[0] if len(rel.parts) > 1 else ""
        row = {"filepath": rel.as_posix(), "class": label, "is_mask": bool(MASK_PATTERN.search(path.stem)),
               "sha256": sha256(path), "status": "ok", "error": "", "width": None,
               "height": None, "mode": "", "channels": None, "phash": "", "mask_count": 0,
               "mean_intensity": None, "std_intensity": None, "dark_fraction": None, "bright_fraction": None}
        try:
            if label not in CLASSES or len(rel.parts) != 2:
                raise ValueError("Unexpected directory; expected class/file.png")
            if path.suffix.lower() != ".png":
                raise ValueError("Expected PNG extension")
            with Image.open(path) as image:
                if image.format != "PNG":
                    raise ValueError("File is not PNG")
                image.load()  # Fully decode; Image.open alone does not detect truncated data.
                if min(image.size) <= 0:
                    raise ValueError("Empty image")
                row.update(width=image.width, height=image.height, mode=image.mode, channels=len(image.getbands()))
                if not row["is_mask"]:
                    row["phash"] = phash(image)
                    gray = np.asarray(image.convert("L"), dtype=float) / 255
                    row.update(mean_intensity=float(gray.mean()), std_intensity=float(gray.std()),
                               dark_fraction=float((gray < 0.02).mean()), bright_fraction=float((gray > 0.98).mean()))
        except (OSError, ValueError, Image.DecompressionBombError) as exc:
            row.update(status="invalid", error=str(exc))
        rows.append(row)
    if not rows:
        raise ValueError("Dataset contains no files")
    frame = pd.DataFrame(rows)
    for idx, row in frame[~frame.is_mask].iterrows():
        image_path = Path(row.filepath)
        masks = frame[frame.is_mask & (frame["class"] == row["class"])]
        frame.loc[idx, "mask_count"] = sum(
            MASK_PATTERN.sub("", Path(p).stem).lower() == image_path.stem.lower()
            for p in masks.filepath
        )
    for label in CLASSES:
        if frame[(frame["class"] == label) & ~frame.is_mask & (frame.status == "ok")].empty:
            raise ValueError(f"No valid original images in class {label}")
    return frame




In [ ]:
def group_images(inventory, patient_csv=None, threshold=4):
    """Connected components of pHash <= threshold, exact hashes and optional patients.

    patient_csv must cover every valid original: filepath,patient_id. A filename
    number is NOT a patient identifier. Mixed-label components need manual review.
    """
    frame = inventory[(inventory.status == "ok") & ~inventory.is_mask].copy()
    frame = frame.sort_values("filepath").reset_index(drop=True)
    parent = list(range(len(frame)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(i, j):
        a, b = find(i), find(j)
        parent[max(a, b)] = min(a, b)

    if patient_csv is not None:
        patients = pd.read_csv(patient_csv, dtype=str, keep_default_na=False)
        if not {"filepath", "patient_id"} <= set(patients.columns):
            raise ValueError("Patient CSV requires filepath,patient_id")
        if patients.filepath.duplicated().any() or patients.patient_id.str.strip().eq("").any():
            raise ValueError("Patient CSV contains duplicate paths or empty IDs")
        mapping = patients.set_index("filepath").patient_id
        frame["patient_id"] = frame.filepath.map(mapping)
        if frame.patient_id.isna().any():
            raise ValueError("Patient CSV must cover every valid original image")
        for indices in frame.groupby("patient_id").groups.values():
            for idx in indices:
                union(indices[0], idx)

    hashes = [int(value, 16) for value in frame.phash]
    pairs = []
    for i in range(len(frame)):
        for j in range(i):
            distance = (hashes[i] ^ hashes[j]).bit_count()
            exact = frame.at[i, "sha256"] == frame.at[j, "sha256"]
            if exact or distance <= threshold:
                union(i, j)
                pairs.append({"filepath_a": frame.at[j, "filepath"], "filepath_b": frame.at[i, "filepath"],
                              "hamming": distance, "exact": exact})
    frame["group_id"] = [f"g{find(i):05d}" for i in range(len(frame))]
    conflicts = frame.groupby("group_id")["class"].nunique()
    frame["needs_review"] = frame.group_id.isin(conflicts[conflicts > 1].index)
    return frame, pd.DataFrame(pairs, columns=["filepath_a", "filepath_b", "hamming", "exact"])


def create_split(grouped, seed=SEED):
    """Stratify homogeneous components; search group splits nearest 70/15/15.

    GroupShuffleSplit samples groups, not images. Candidate selection minimizes
    image-count error within each class. Indivisible groups imply approximate ratios.
    """
    frame = grouped[~grouped.needs_review].copy().sort_values("filepath").reset_index(drop=True)
    frame["split"] = ""
    targets = np.array([0.70, 0.15, 0.15])
    for label in CLASSES:
        subset = frame[frame["class"] == label]
        if subset.group_id.nunique() < 3:
            raise ValueError(f"{label}: need >=3 independent groups after quarantine; inspect conflicts")
        best, best_score = None, float("inf")
        group_count = subset.group_id.nunique()
        # Leave at least two groups for validation and test, including tiny fixtures.
        train_fraction = min(0.7, (group_count - 2) / group_count + 1e-9)
        first = GroupShuffleSplit(n_splits=128, train_size=train_fraction, random_state=seed)
        for train_idx, rest_idx in first.split(subset, groups=subset.group_id):
            rest = subset.iloc[rest_idx]
            if rest.group_id.nunique() < 2:
                continue
            second = GroupShuffleSplit(n_splits=32, test_size=0.5, random_state=seed)
            for val_idx, test_idx in second.split(rest, groups=rest.group_id):
                parts = (subset.iloc[train_idx].index, rest.iloc[val_idx].index, rest.iloc[test_idx].index)
                score = float(np.square(np.array([len(p) for p in parts]) / len(subset) - targets).sum())
                if score < best_score:
                    best, best_score = parts, score
        if best is None:
            raise ValueError(f"Cannot split {label}; inspect group sizes")
        for name, indices in zip(("train", "validation", "test"), best):
            frame.loc[indices, "split"] = name
    validate_split(frame)
    return frame[["filepath", "class", "group_id", "split"]]


def validate_split(frame):
    if frame.filepath.duplicated().any() or frame.empty:
        raise ValueError("Duplicate paths or empty split")
    if set(frame["class"]) != set(CLASSES) or set(frame.split) != {"train", "validation", "test"}:
        raise ValueError("Invalid class/split names")
    if (frame.groupby("group_id").split.nunique() != 1).any():
        raise ValueError("Group leakage across splits")
    if (pd.crosstab(frame["class"], frame.split).reindex(CLASSES, fill_value=0) == 0).any().any():
        raise ValueError("Every split must contain all three classes")


def freeze_split(grouped, inventory, output_dir, version="v1", seed=SEED, threshold=4):
    """Never silently overwrite a split; verify bytes, membership and data on rerun."""
    if not re.fullmatch(r"v[1-9]\d*", version):
        raise ValueError("Use split version v1, v2, ...")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / f"split_{version}.csv"
    metadata_path = path.with_suffix(".json")
    fingerprint_columns = ["filepath", "class", "sha256", "status", "is_mask"]
    dataset_hash = hashlib.sha256(inventory.sort_values("filepath")[fingerprint_columns].to_csv(index=False, lineterminator="\n").encode()).hexdigest()
    grouping_columns = ["filepath", "class", "group_id", "needs_review"]
    grouping_hash = hashlib.sha256(grouped.sort_values("filepath")[grouping_columns].to_csv(index=False, lineterminator="\n").encode()).hexdigest()
    identity = {"dataset_sha256": dataset_hash, "grouping_sha256": grouping_hash,
                "seed": seed, "phash_threshold": threshold, "class_to_idx": CLASS_TO_IDX,
                "split_algorithm": "per-class-group-search-128x32-v1", "ratios": [0.7, 0.15, 0.15]}
    if path.exists() or metadata_path.exists():
        if not (path.exists() and metadata_path.exists()):
            raise ValueError("Incomplete split artifacts; restore matching CSV and JSON")
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if any(metadata.get(key) != value for key, value in identity.items()) or sha256(path) != metadata["split_sha256"]:
            raise ValueError("Data, grouping, config or split changed. Restore artifacts or use a NEW version.")
        result = pd.read_csv(path, dtype=str)
        validate_split(result)
        expected = grouped[~grouped.needs_review][["filepath", "class", "group_id"]].sort_values("filepath").reset_index(drop=True)
        if not result.sort_values("filepath")[expected.columns].reset_index(drop=True).equals(expected):
            raise ValueError("Split membership differs from audited data")
        return result, metadata
    result = create_split(grouped, seed)
    result.to_csv(path, index=False, lineterminator="\n")
    metadata = {**identity, "split_sha256": sha256(path), "environment": environment(),
                "source": DATASET_SLUG, "patient_ids_available": "patient_id" in grouped.columns,
                "limitations": "pHash groups are not patients; patient leakage remains possible without reliable IDs."}
    write_json(metadata_path, metadata)
    return result, metadata




In [ ]:
def train_statistics(root, split):
    """Pixel-weighted mean/std of RGB [0,1] after resize, TRAIN only, no augment."""
    total, squares, pixels = np.zeros(3), np.zeros(3), 0
    train = split[split.split == "train"]
    if train.empty:
        raise ValueError("Empty training split")
    for relative in train.filepath:
        with Image.open(Path(root) / relative) as image:
            values = np.asarray(image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR), dtype=np.float64) / 255
        total += values.sum(axis=(0, 1))
        squares += np.square(values).sum(axis=(0, 1))
        pixels += values.shape[0] * values.shape[1]
    mean = total / pixels
    std = np.sqrt(np.maximum(squares / pixels - mean ** 2, 0))
    if (std < 1e-8).any():
        raise ValueError("Zero variance in training channels; inspect dataset")
    return {"mean": mean.tolist(), "std": std.tolist(), "images": len(train), "pixels": pixels}


def preprocessing_config(stats, split_sha256):
    return {"split_sha256": split_sha256, "size": [IMAGE_SIZE, IMAGE_SIZE], "channels": 3,
            "color_mode": "RGB", "resize_interpolation": "bilinear", "scale": "[0,1]",
            "class_to_idx": CLASS_TO_IDX,
            "baseline": {"mean": stats["mean"], "std": stats["std"]},
            "imagenet": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
            "augmentation": {"horizontal_flip_p": 0.5, "rotation_degrees": 10,
                             "crop_scale": [0.85, 1.0], "crop_ratio": [1.0, 1.0],
                             "brightness": 0.1, "contrast": 0.1}}


def convert_rgb(image):
    """Named function so transforms remain picklable for Windows workers."""
    return image.convert("RGB")


def make_transform(config, split="validation", normalization="baseline"):
    from torchvision import transforms as T

    if split not in ("train", "validation", "test"):
        raise ValueError("Unknown split")
    if normalization not in ("baseline", "imagenet"):
        raise ValueError("Unknown normalization")
    size = tuple(config["size"])
    # Resize is shared with train_statistics; stochastic transforms follow resize.
    operations = [T.Lambda(convert_rgb), T.Resize(size, interpolation=T.InterpolationMode.BILINEAR, antialias=True)]
    if split == "train":
        aug = config["augmentation"]
        operations += [T.RandomHorizontalFlip(aug["horizontal_flip_p"]),
                       T.RandomRotation(aug["rotation_degrees"], interpolation=T.InterpolationMode.BILINEAR),
                       T.RandomResizedCrop(size, scale=tuple(aug["crop_scale"]), ratio=tuple(aug["crop_ratio"]),
                                           interpolation=T.InterpolationMode.BILINEAR, antialias=True),
                       T.ColorJitter(brightness=aug["brightness"], contrast=aug["contrast"])]
    parameters = config[normalization]
    return T.Compose(operations + [T.ToTensor(), T.Normalize(parameters["mean"], parameters["std"])])


def seed_everything(seed=SEED):
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)


def seed_worker(worker_id):
    import torch

    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loaders(root, split, config, normalization="baseline", batch_size=32, seed=SEED, num_workers=0):
    import torch
    from torch.utils.data import DataLoader

    validate_split(split)
    loaders = {}
    for name in ("train", "validation", "test"):
        dataset = BUSIDataset(root, split[split.split == name], make_transform(config, name, normalization))
        generator = torch.Generator().manual_seed(seed)
        loaders[name] = DataLoader(dataset, batch_size=batch_size, shuffle=name == "train",
                                   num_workers=num_workers, worker_init_fn=seed_worker, generator=generator)
    return loaders


class BUSIDataset:
    """Map-style PyTorch dataset; no masks or patient identifiers enter the model."""

    def __init__(self, root, rows, transform):
        self.root = Path(root).resolve()
        self.rows = rows.reset_index(drop=True).copy()
        self.transform = transform
        for relative in self.rows.filepath:
            path = (self.root / relative).resolve()
            if not path.is_relative_to(self.root) or MASK_PATTERN.search(path.stem):
                raise ValueError("Unsafe path or mask in classification manifest")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        with Image.open(self.root / row.filepath) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, CLASS_TO_IDX[row["class"]]


def class_weights(split):
    counts = split[split.split == "train"]["class"].value_counts().reindex(CLASSES, fill_value=0)
    if (counts == 0).any():
        raise ValueError("Training data missing a class")
    return (counts.sum() / (len(CLASSES) * counts)).to_numpy(dtype=np.float32)


In [ ]:
# Настройки Colab: только пути/режим хранения; код проекта загружать не нужно.
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

WORKSPACE = (Path("/content/drive/MyDrive/breast-cancer-classification")
             if USE_GOOGLE_DRIVE else Path("/content/breast-cancer-classification"))
ARTIFACTS = WORKSPACE / "artifacts"
KAGGLE_VERSION = 1
SPLIT_VERSION = "v1"
# Папка с классами normal/benign/malignant или None для загрузки ZIP.
LOCAL_DATA_ROOT = None
# Можно заранее положить ZIP из Kaggle сюда; установка Kaggle API не нужна.
DATASET_ZIP = WORKSPACE / f"raw/busi-kaggle-v{KAGGLE_VERSION}.zip"
# Необязательные реальные patient ID: filepath,patient_id (пути относительно DATA_ROOT).
PATIENT_CSV = None
RUN_DIR = ARTIFACTS / "data" / SPLIT_VERSION
RUN_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(SEED)
print("Артефакты:", ARTIFACTS)
if not USE_GOOGLE_DRIVE:
    print("Внимание: /content временный. Скачайте artifacts перед завершением сессии.")


In [ ]:
# Загрузка публичного ZIP стандартной библиотекой или подключение готовой папки.
# Если Kaggle потребует вход, скачайте ZIP на странице датасета и поместите
# в DATASET_ZIP на Drive, либо задайте LOCAL_DATA_ROOT. Токены в код не нужны.
if LOCAL_DATA_ROOT is None:
    DATASET_ZIP.parent.mkdir(parents=True, exist_ok=True)
    if not DATASET_ZIP.exists():
        url = ("https://www.kaggle.com/api/v1/datasets/download/"
               f"{DATASET_SLUG}?datasetVersionNumber={KAGGLE_VERSION}")
        temporary_zip = DATASET_ZIP.with_suffix(".download")
        request = urllib.request.Request(url, headers={"User-Agent": "BUSI-Colab/1.0"})
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                with temporary_zip.open("wb") as output:
                    shutil.copyfileobj(response, output)
            if not zipfile.is_zipfile(temporary_zip):
                raise ValueError("Kaggle вернул не ZIP")
            temporary_zip.replace(DATASET_ZIP)
        except Exception as exc:
            temporary_zip.unlink(missing_ok=True)
            raise RuntimeError(
                "Не удалось загрузить публичный ZIP. Скачайте версию BUSI с Kaggle "
                "в DATASET_ZIP или задайте LOCAL_DATA_ROOT и повторите эту ячейку."
            ) from exc
    archive_sha256 = sha256(DATASET_ZIP)
    extraction = Path("/content/busi-data") / archive_sha256
    extraction.mkdir(parents=True, exist_ok=True)
    marker = extraction.with_suffix(".complete")
    if not marker.exists():
        with zipfile.ZipFile(DATASET_ZIP) as archive:
            # Не допускаем запись ZIP за пределами выделенного каталога.
            for member in archive.infolist():
                target = (extraction / member.filename).resolve()
                if not target.is_relative_to(extraction.resolve()):
                    raise ValueError("Небезопасный путь в ZIP")
            archive.extractall(extraction)
        marker.touch()
    DATA_ROOT = find_data_root(extraction)
    source = {"dataset": DATASET_SLUG, "requested_kaggle_version": KAGGLE_VERSION,
              "archive_sha256": archive_sha256}
else:
    DATA_ROOT = find_data_root(LOCAL_DATA_ROOT)
    source = {"dataset": DATASET_SLUG, "version": "local-user-supplied"}
write_json(RUN_DIR / "source.json",
           {**source, "pipeline_version": PIPELINE_VERSION, "environment": environment()})
print("Dataset:", DATA_ROOT)
display(pd.Series(environment()))


## 2. Аудит структуры и качества

Все файлы попадут в inventory.csv с SHA-256, статусом декодирования, размерами и режимом каналов. Маски `_mask.png`, `_mask_1.png` и т. д. распознаются отдельно и проверяются, но исключаются из input pipeline. Повреждённые оригиналы исключаются с явным отчётом. Чужие файлы и вложенные папки отмечаются invalid.

Показатели яркости и контраста помогают выбрать кадры для просмотра; они не служат автоматическим медицинским фильтром. Отсутствие маски отмечается, но не мешает классификации.

In [ ]:
inventory = scan_dataset(DATA_ROOT)
inventory.to_csv(RUN_DIR / "inventory.csv", index=False)
invalid = inventory[inventory.status != "ok"]
invalid.to_csv(RUN_DIR / "invalid_files.csv", index=False)
originals = inventory[(inventory.status == "ok") & ~inventory.is_mask].copy()
summary = inventory.groupby(["class", "is_mask", "status"], dropna=False).size().rename("files").reset_index()
summary.to_csv(RUN_DIR / "audit_summary.csv", index=False)
display(summary)
display(invalid[["filepath", "error"]])
display(originals.groupby("class")[["width", "height", "channels", "mask_count"]].describe())
print("Оригиналы без масок:", int((originals.mask_count == 0).sum()))
print("Невалидных файлов:", len(invalid))


## 3. EDA: классы, каналы, размеры и примеры

Графики автоматически сохраняются в PNG для отчёта. Просмотрите подписи/рамки, встроенный текст, засветку и чёрные поля. Не меняйте pipeline по качеству на test: EDA ниже описывает исходный датасет до экспериментов, а подбор преобразований в дальнейшем выполняется только по train/validation.

In [ ]:
def save_figure(name):
    plt.tight_layout()
    plt.savefig(RUN_DIR / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()

counts = originals["class"].value_counts().reindex(CLASSES, fill_value=0)
class_report = pd.DataFrame({"count": counts, "fraction": counts / counts.sum()})
class_report.to_csv(RUN_DIR / "class_distribution.csv")
display(class_report)
print("Дисбаланс max/min:", round(counts.max() / counts.min(), 3))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
counts.plot.bar(ax=axes[0], title="Классы (оригиналы)", rot=0)
axes[1].scatter(originals.width, originals.height, alpha=0.45)
axes[1].set(xlabel="Ширина, px", ylabel="Высота, px", title="Размеры")
originals["mode"].value_counts().plot.bar(ax=axes[2], title="Режим каналов", rot=0)
save_figure("eda_distribution")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, field in zip(axes, ["mean_intensity", "std_intensity", "dark_fraction"]):
    for label in CLASSES:
        axis.hist(originals.loc[originals["class"] == label, field], bins=25, alpha=0.4, label=label)
    axis.set_title(field)
    axis.legend()
save_figure("eda_intensity")


In [ ]:
def gallery(rows, name, columns=3):
    rows = rows.reset_index(drop=True)
    if rows.empty:
        print(name, "— нет изображений")
        return
    nrows = (len(rows) + columns - 1) // columns
    fig, axes = plt.subplots(nrows, columns, figsize=(4 * columns, 3.5 * nrows), squeeze=False)
    for axis in axes.ravel():
        axis.axis("off")
    for axis, row in zip(axes.ravel(), rows.itertuples()):
        with Image.open(DATA_ROOT / row.filepath) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(row.filepath, fontsize=8)
    save_figure(name)

examples = pd.concat([originals[originals["class"] == c].sample(min(3, int(counts[c])), random_state=SEED)
                      for c in CLASSES])
gallery(examples, "eda_examples")
suspects = pd.concat([originals.nsmallest(3, "std_intensity"),
                      originals.nlargest(3, "bright_fraction"),
                      originals.nlargest(3, "dark_fraction")]).drop_duplicates("filepath")
gallery(suspects, "eda_quality_review")


### Ручные наблюдения после запуска

Заполните после просмотра: какие артефакты найдены; какие файлы требуют исключения и почему; вывод о дисбалансе. Не удаляйте исходники молча. Если решения меняют состав уже зафиксированного split, нужна новая версия и повторное обучение всех моделей.

## 4. Дубликаты и группы

64-bit pHash: grayscale → 32×32 → DCT → 8×8 → порог медианы. Рёбра при расстоянии Хэмминга ≤4 или одинаковом SHA-256; связные компоненты сохраняют и транзитивные связи. Для малого BUSI используется прозрачный O(N²) поиск пар.

Если есть надёжные patient ID, передайте PATIENT_CSV: `filepath,patient_id`, путь относительно DATA_ROOT; CSV должен покрывать каждый валидный оригинал. Номер в имени файла **не** считается patient ID. Связи пациенток объединяются с дубликатами. Любая компонента с разными метками консервативно исключается целиком до ручного решения.

Без patient ID нельзя гарантировать отсутствие patient leakage: pHash контролирует лишь похожие снимки. Даже при группировке метрики могут быть завышены.

In [ ]:
grouped, duplicate_pairs = group_images(inventory, PATIENT_CSV, threshold=4)
grouped.to_csv(RUN_DIR / "grouped_inventory.csv", index=False)
duplicate_pairs.to_csv(RUN_DIR / "duplicate_pairs.csv", index=False)
conflicts = grouped[grouped.needs_review]
conflicts.to_csv(RUN_DIR / "conflicts.csv", index=False)
display(duplicate_pairs.head(20))
display(conflicts[["filepath", "class", "group_id"]])
print("Валидных оригиналов:", len(grouped), "групп:", grouped.group_id.nunique())
print("Точных пар:", int(duplicate_pairs.exact.sum()), "всех похожих пар:", len(duplicate_pairs))
print("Карантин:", len(conflicts), "изображений")
gallery(conflicts.head(12), "eda_conflicts")
if not duplicate_pairs.empty:
    review_paths = pd.unique(duplicate_pairs.head(6)[["filepath_a", "filepath_b"]].to_numpy().ravel())
    gallery(grouped[grouped.filepath.isin(review_paths)], "eda_duplicate_examples")


## 5. Стратифицированный group split 70/15/15, seed 42

Все компоненты однородны после карантина. Для каждого класса выбирается ближайшее по количеству изображений разделение из фиксированных кандидатов GroupShuffleSplit (128×32). Группы неделимы, поэтому доли приблизительны; при нехватке трёх независимых групп выполнение остановится.

split_v1.csv хранит `filepath,class,group_id,split`. JSON фиксирует SHA-256 split, fingerprint всех файлов (включая маски/битые), группировку, параметры, источник и версии библиотек. Повторный запуск загружает тот же split, сверяет содержимое и отказывается перезаписывать изменённые данные. Новые данные/решения → новая версия v2, а не ещё один случайный split.

In [ ]:
split, split_metadata = freeze_split(
    grouped, inventory, ARTIFACTS / "splits", version=SPLIT_VERSION, seed=SEED, threshold=4,
)
validate_split(split)
assert set(split.filepath) == set(grouped.loc[~grouped.needs_review, "filepath"])
assert split.groupby("group_id").split.nunique().max() == 1
assert not split.filepath.str.contains(r"_mask", case=False).any()
if "patient_id" in grouped:
    joined = split.merge(grouped[["filepath", "patient_id"]], on="filepath", validate="one_to_one")
    assert joined.groupby("patient_id").split.nunique().max() == 1
split_counts = pd.crosstab(split["class"], split.split).reindex(index=CLASSES, columns=["train", "validation", "test"])
split_counts.to_csv(RUN_DIR / "split_counts.csv")
display(split_counts)
display(split_counts.div(split_counts.sum(axis=1), axis=0).round(3))
split_counts.plot.bar(stacked=True, figsize=(8, 4), rot=0, title="Состав split")
save_figure("split_distribution")
print("Split SHA-256:", split_metadata["split_sha256"])
display(pd.Series(split_metadata))


## 6. Train-only statistics и preprocessing

224×224, RGB (grayscale дублируется по каналам), bilinear resize, tensor [0,1]. Mean/std baseline считаются исключительно по пикселям train после детерминированного resize. Для pretrained CNN/ViT доступна фиксированная ImageNet-нормализация: проверьте её соответствие конкретным выбранным весам в следующем эксперименте.

Только train: horizontal flip p=0.5, rotation ±10°, crop scale 0.85–1.0 без искажения aspect ratio, brightness/contrast ±10%. Вертикальный flip и hue/saturation отсутствуют. Validation/test всегда детерминированы. Config и split SHA сохраняйте в checkpoint; class weights вычисляются только по train.

In [ ]:
stats = train_statistics(DATA_ROOT, split)
config = preprocessing_config(stats, split_metadata["split_sha256"])
write_json(RUN_DIR / "train_statistics.json", stats)
write_json(RUN_DIR / "preprocessing.json", config)
weights = class_weights(split)
write_json(RUN_DIR / "class_weights.json", dict(zip(CLASSES, map(float, weights))))
display(pd.DataFrame(config["baseline"], index=["R", "G", "B"]))
display(pd.Series(dict(zip(CLASSES, weights)), name="train class weight"))

sample_path = split.loc[split.split == "train", "filepath"].iloc[0]
with Image.open(DATA_ROOT / sample_path) as image:
    sample = image.convert("RGB")
transform = make_transform(config, "train")
mean = np.array(config["baseline"]["mean"])
std = np.array(config["baseline"]["std"])
seed_everything(SEED)
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
axes[0].imshow(sample)
axes[0].set_title("Оригинал train")
for axis in axes[1:]:
    tensor = transform(sample)
    axis.imshow(np.clip(tensor.permute(1, 2, 0).numpy() * std + mean, 0, 1))
    axis.set_title("Train augmentation")
for axis in axes:
    axis.axis("off")
save_figure("train_augmentations")


## 7. Dataset/DataLoader и проверки контракта

Ниже выполняется только smoke check input pipeline, без обучения и подсчёта model metrics. num_workers=0 подходит для Windows/Colab; worker seeds поддерживаются при увеличении числа workers. Для повторения batches создавайте новые loaders с тем же seed и вызывайте seed_everything перед итерацией. Сохранённый split не зависит от seed обучения.

In [ ]:
import torch

seed_everything(SEED)
loaders = make_loaders(DATA_ROOT, split, config, normalization="baseline", seed=SEED, num_workers=0)
images, labels = next(iter(loaders["train"]))
assert images.ndim == 4 and tuple(images.shape[1:]) == (3, IMAGE_SIZE, IMAGE_SIZE)
assert torch.isfinite(images).all() and set(labels.tolist()) <= {0, 1, 2}
for name in ("validation", "test"):
    dataset = loaders[name].dataset
    assert torch.equal(dataset[0][0], dataset[0][0])
print("Batch:", tuple(images.shape), "labels:", labels.tolist())
print("Input pipeline smoke checks: OK")
print("Loss weights:", torch.tensor(weights))
# Для ResNet/ViT:
# loaders = make_loaders(DATA_ROOT, split, config, normalization="imagenet", seed=43)


## 8. Версионирование и передача в обучение

Inventory хранит SHA-256 каждого raw-файла; split JSON связывает dataset fingerprint, группировку и split. Итоговый manifest связывает отчёты и preprocessing с версией кода notebook и фактическими версиями библиотек Colab. Среда Colab обновляется, поэтому межплатформенное численное совпадение без сохранённого окружения не гарантируется. Установка фиксированных версий не выполняется.

По умолчанию ZIP, split и отчёты сохраняются на Google Drive. Манифесты проверяют целостность; **сохраните этот .ipynb в Drive вместе с artifacts**. При работе без Drive скачайте artifacts до завершения сессии. DVC/remote не требуются.

Будущие эксперименты должны загрузить сохранённый split, а не генерировать его заново. SHA-256 split и preprocessing записываются в checkpoint. Test используется только для итогового сравнения: один запуск на модель × seed. Здесь выполняется только E0.

In [ ]:
# Контракт будущих экспериментов из UNI-32/33; результаты сюда не подставляются.
experiment_contract = {
    "design_issues": ["UNI-23", "UNI-31", "UNI-32", "UNI-33"],
    "stage": "E0: data preparation; no model training/evaluation",
    "model_families": ["baseline CNN", "transfer CNN (ResNet-18)", "transfer ViT tiny/small"],
    "training_seeds": [42, 43, 44],
    "split_sha256": split_metadata["split_sha256"],
    "primary_metric": "macro-F1",
    "required_metrics": ["Accuracy", "per-class Precision/Recall/F1", "confusion matrix",
                         "macro ROC-AUC OvR", "CPU latency mean/p95", "checkpoint size"],
    "thresholds": {"Accuracy": 0.80, "macro-F1": 0.75, "Recall malignant": 0.80,
                   "macro ROC-AUC OvR": 0.90, "min per-class F1": 0.60},
    "preliminary_thresholds": ["macro-F1", "Recall malignant", "macro ROC-AUC OvR", "min per-class F1"],
    "threshold_revision": "Once after baseline and before other models; record justification in UNI-32",
    "validation_role": "early stopping, checkpoint and hyperparameter selection",
    "test_role": "one final evaluation per model and seed",
}
write_json(RUN_DIR / "experiment_contract.json", experiment_contract)
manifest = {
    "schema_version": 1, "source": source, "pipeline_version": PIPELINE_VERSION,
    "split_version": SPLIT_VERSION, "split_sha256": split_metadata["split_sha256"],
    "dataset_sha256": split_metadata["dataset_sha256"],
    "environment": environment(),
    "files": {
        p.relative_to(ARTIFACTS).as_posix(): sha256(p)
        for p in sorted(RUN_DIR.rglob("*"))
        if p.is_file() and p.name != "manifest.json"
    },
}
for suffix in ("csv", "json"):
    p = ARTIFACTS / "splits" / f"split_{SPLIT_VERSION}.{suffix}"
    manifest["files"][p.relative_to(ARTIFACTS).as_posix()] = sha256(p)
write_json(RUN_DIR / "manifest.json", manifest)
assert all(sha256(ARTIFACTS / name) == digest for name, digest in manifest["files"].items())
print("Готово:", RUN_DIR)
print("Сохраните notebook и artifacts в Drive. Метрики моделей здесь не вычислялись.")


### Что проверить вручную

- Проверить invalid_files.csv, конфликты меток и gallery похожих снимков.
- Описать артефакты качества и дисбаланс в разделе ручных наблюдений.
- Проверить фактические доли split и отсутствие разнесённых patient/group ID.
- Убедиться, что умеренные аугментации не нарушают структуру снимков.
- Сохранить artifacts и результаты своего запуска; повторить запуск с теми же данными.
- Отдельно задокументировать принятые решения по карантину перед началом E1.

Ограничения: нет клинической валидации; pHash даёт false positives/negatives, качество групп требует ручного просмотра. Доли могут отличаться от целевых из-за размеров групп. Воспроизводимость чисел между платформами требует совпадающих версий зависимостей.